# HybridCC VOD — Colab notebook

End-to-end test of the VOD CEA-608 pipeline.

```
input.mp4 ─► stable-ts (faster-whisper) ─► input.vtt
                                              │
input.mp4 ──┬─────────────────────────────────┘
            ▼
         ffmpeg (mp4 → flv pipe)
            │
            ▼
         hybridCC-vod  ◄── input.vtt   (PTS-matched CEA-608 SEI)
            │
            ▼
         ffmpeg (flv → mp4, -a53cc 1)
            │
            ▼
         output.mp4 with embedded CEA-608
```

**Runtime:** GPU (free-tier T4 is enough). Run cells top-to-bottom.

## 1 · GPU + disk check

In [17]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — Runtime → Change runtime type → GPU'
!df -h / | tail -1

Tesla T4, 15360 MiB
overlay         113G   47G   67G  42% /


## 2 · Install build deps + ffmpeg

Colab images already have build-essential + cmake. ffmpeg is preinstalled. re2c is optional but available.

In [18]:
!apt-get -qq install -y build-essential cmake git ffmpeg re2c 2>&1 | tail -3
!gcc --version | head -1
!cmake --version | head -1
!ffmpeg -version 2>&1 | head -1

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
cmake version 3.31.10
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


## 3 · Build libcaption (upstream, MIT)

In [19]:
%cd /content
![ -d libcaption ] || git clone --depth 1 https://github.com/szatmary/libcaption.git
%cd /content/libcaption
# Verbose — show actual cmake/make output so we can see failures
!cmake . -DENABLE_RE2C=ON 2>&1 | tail -15
!echo '--- make ---'
!make -j$(nproc) 2>&1 | tail -15
!echo '\n--- build artifacts (.a / .so) ---'
!find . -maxdepth 3 \( -name '*.a' -o -name '*.so*' \) 2>/dev/null
!echo '\n--- src/ .c files (used by fallback compile if no lib found) ---'
!ls src/*.c 2>/dev/null | head -20

/content/libcaption
/content/libcaption
CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- Found re2c: /usr/bin/re2c
-- Could NOT find Doxygen (missing: DOXYGEN_EXECUTABLE) 
-- Configuring done (0.0s)
-- Generating done (0.1s)
-- Build files have been written to: /content/libcaption
--- make ---
[ 60%] Linking C executable flv+scc
[ 64%] Built target flv+srt
[ 66%] Linking C executable sccdump
[ 70%] Built target flv+scc
[ 72%] Linking C executable srtdump
[ 77%] Built target sccdump
[ 81%] Built target srtdump
[ 81%] Linking C executable vttdump
[ 83%] Linking C executable rollup
[ 85%] Built target vttdump
[ 87%] Linking C executable party
[ 91%] Built target rollup
[

## 4 · Write & compile `hybridCC-vod.c`

Embedded source for v1. Once `hybridcc/` is on GitHub, replace the heredoc with `git clone`.

In [20]:
from pathlib import Path

VOD_C = r'''
/*
 * hybridCC-vod -- CEA-608 caption injector for VOD (timestamp-matched)
 * Reads FLV from stdin, parses VTT cues, injects CEA-608 SEI by PTS.
 * Writes captioned FLV to stdout.
 *
 * Injects on STATE CHANGE only (when active cue text changes), not every
 * frame. Avoids the "90 redundant captions per cue" extraction noise.
 */

#include "caption/caption.h"
#include "flv.h"
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#ifdef _WIN32
#include <io.h>
#include <fcntl.h>
#endif

#define MAX_VTT_SIZE  (4 * 1024 * 1024)
#define MAX_CUES       20000
#define MAX_CUE_TEXT   1024

typedef struct {
    double start;
    double end;
    char   text[MAX_CUE_TEXT];
} vtt_cue_t;

static vtt_cue_t g_cues[MAX_CUES];
static int g_cue_count = 0;
static int g_cursor    = 0;

static double parse_ts(const char* s, int len)
{
    char buf[32];
    if (len <= 0 || len >= (int)sizeof(buf)) return -1.0;
    memcpy(buf, s, len);
    buf[len] = 0;
    int h = 0, m = 0, sec = 0, ms = 0;
    if (sscanf(buf, "%d:%d:%d.%d", &h, &m, &sec, &ms) == 4)
        return h * 3600.0 + m * 60.0 + sec + ms / 1000.0;
    if (sscanf(buf, "%d:%d.%d", &m, &sec, &ms) == 3)
        return m * 60.0 + sec + ms / 1000.0;
    return -1.0;
}

static int load_vtt_cues(const char* path)
{
    FILE* f = fopen(path, "rb");
    if (!f) { fprintf(stderr, "[CC] Cannot open VTT: %s\n", path); return 0; }
    fseek(f, 0, SEEK_END);
    long sz = ftell(f);
    fseek(f, 0, SEEK_SET);
    if (sz <= 0 || sz >= MAX_VTT_SIZE) { fclose(f); return 0; }
    char* buf = (char*)malloc((size_t)sz + 1);
    if (!buf) { fclose(f); return 0; }
    if (fread(buf, 1, (size_t)sz, f) != (size_t)sz) { free(buf); fclose(f); return 0; }
    buf[sz] = 0;
    fclose(f);

    char* p = buf;
    while (*p && g_cue_count < MAX_CUES) {
        char* arrow = strstr(p, "-->");
        if (!arrow) break;
        char* line_start = arrow;
        while (line_start > buf && line_start[-1] != '\n') line_start--;
        char* s_end = arrow;
        while (s_end > line_start && (s_end[-1] == ' ' || s_end[-1] == '\t')) s_end--;
        double start = parse_ts(line_start, (int)(s_end - line_start));
        char* e_start = arrow + 3;
        while (*e_start == ' ' || *e_start == '\t') e_start++;
        char* e_end = e_start;
        while (*e_end && *e_end != ' ' && *e_end != '\r' && *e_end != '\n') e_end++;
        double end = parse_ts(e_start, (int)(e_end - e_start));
        if (start < 0 || end <= start) { p = arrow + 3; continue; }

        char* text_start = strchr(arrow, '\n');
        if (!text_start) break;
        text_start++;

        char* text_end = strstr(text_start, "\r\n\r\n");
        if (!text_end) text_end = strstr(text_start, "\n\n");
        char* next_arrow = strstr(text_start, "-->");
        if (next_arrow) {
            char* nal = next_arrow;
            while (nal > text_start && nal[-1] != '\n') nal--;
            if (!text_end || nal < text_end) text_end = nal;
        }
        if (!text_end) text_end = buf + sz;

        vtt_cue_t* cue = &g_cues[g_cue_count];
        int outlen = 0, prev_space = 1;
        int span = (int)(text_end - text_start);
        for (int i = 0; i < span; i++) {
            char c = text_start[i];
            if (c == '\r' || c == '\n' || c == '\t') c = ' ';
            if (c == ' ' && prev_space) continue;
            if (outlen >= MAX_CUE_TEXT - 1) break;
            cue->text[outlen++] = c;
            prev_space = (c == ' ');
        }
        while (outlen > 0 && cue->text[outlen-1] == ' ') outlen--;
        cue->text[outlen] = 0;
        if (outlen > 0) {
            cue->start = start;
            cue->end   = end;
            g_cue_count++;
        }
        p = text_end;
    }
    free(buf);
    fprintf(stderr, "[CC] Parsed %d cues from %s\n", g_cue_count, path);
    return g_cue_count;
}

static const char* find_cue_at(double pts_sec)
{
    while (g_cursor < g_cue_count && g_cues[g_cursor].end <= pts_sec) g_cursor++;
    if (g_cursor >= g_cue_count) return NULL;
    if (pts_sec >= g_cues[g_cursor].start) return g_cues[g_cursor].text;
    return NULL;
}

int main(int argc, char** argv)
{
    if (argc < 2) {
        fprintf(stderr, "Usage: %s captions.vtt < input.flv > output.flv\n", argv[0]);
        return 1;
    }
#ifdef _WIN32
    _setmode(_fileno(stdin), _O_BINARY);
    _setmode(_fileno(stdout), _O_BINARY);
#endif
    load_vtt_cues(argv[1]);
    flvtag_t tag;
    flvtag_init(&tag);
    int has_audio = 0, has_video = 0;
    if (!flv_read_header(stdin, &has_audio, &has_video)) {
        fprintf(stderr, "[CC] Not a valid FLV on stdin\n");
        return 1;
    }
    flv_write_header(stdout, has_audio, has_video);
    fprintf(stderr, "[CC] Streaming (audio=%d video=%d)...\n", has_audio, has_video);

    long video_frames = 0, injected = 0;
    const char* last_text = NULL;   /* track state — only inject on transition */

    while (flv_read_tag(stdin, &tag)) {
        if (flvtag_avcpackettype_nalu == flvtag_avcpackettype(&tag)) {
            video_frames++;
            uint32_t pts_ms = flvtag_pts(&tag);
            const char* text = find_cue_at(pts_ms / 1000.0);

            /* Only inject on cue transition. CEA-608 is stateful — receivers
             * hold the display until told otherwise. Re-injecting every frame
             * makes ffmpeg subcc + CCExtractor dump 90 cues per real cue. */
            if (text != last_text) {
                if (text && *text) {
                    flvtag_addcaption_text(&tag, (const utf8_char_t*)text);
                    injected++;
                }
                last_text = text;
            }
        }
        flv_write_tag(stdout, &tag);
    }
    fprintf(stderr, "[CC] Done: %ld frames, %ld inject events\n", video_frames, injected);
    flvtag_free(&tag);
    return 0;
}
'''

Path('/content/hybridCC-vod.c').write_text(VOD_C)
print('wrote /content/hybridCC-vod.c -', len(VOD_C), 'chars')

wrote /content/hybridCC-vod.c - 5832 chars


In [21]:
import subprocess, glob, os, sys

os.chdir('/content')

# libcaption layout (upstream szatmary/libcaption):
#   src/*.c    → compiled into libcaption.a
#   examples/  → contains flv.h and flv.c (NOT in the .a). Must be compiled
#                alongside any FLV-using example.
#   caption/   → public headers; -I libcaption/caption needed for internal
#                sibling-header #includes
libs = glob.glob('libcaption/libcaption.a') + glob.glob('libcaption/build/libcaption.a') \
     + glob.glob('libcaption/libcaption.so*') + glob.glob('libcaption/build/libcaption.so*')

base = ['gcc', '-O2', '-Wall',
        '-I', 'libcaption',
        '-I', 'libcaption/src',
        '-I', 'libcaption/examples',
        '-I', 'libcaption/caption',
        '-o', 'hybridCC-vod',
        'hybridCC-vod.c',
        'libcaption/examples/flv.c']    # always required — not in libcaption.a

if libs:
    print(f'[build] linking against {libs[0]}')
    cmd = base + [libs[0], '-lm']
else:
    src_files = sorted(glob.glob('libcaption/src/*.c'))
    if not src_files:
        sys.exit('[build] FAIL: no libcaption.a AND no libcaption/src/*.c — clone may have failed')
    print(f'[build] no static lib found — compiling {len(src_files)} libcaption sources directly')
    cmd = base + src_files + ['-lm']

print('[build] cmd:', ' '.join(cmd[:8]), '...')
r = subprocess.run(cmd, capture_output=True, text=True)
if r.stdout: print(r.stdout)
if r.stderr: print(r.stderr)
if r.returncode != 0:
    sys.exit(f'[build] FAIL exit={r.returncode}')

print('\n[build] OK')
subprocess.run(['ls', '-la', '/content/hybridCC-vod'])
print('\n[build] usage smoke test:')
subprocess.run(['/content/hybridCC-vod'], stderr=subprocess.STDOUT)

[build] linking against libcaption/libcaption.a
[build] cmd: gcc -O2 -Wall -I libcaption -I libcaption/src -I ...

[build] OK

[build] usage smoke test:


CompletedProcess(args=['/content/hybridCC-vod'], returncode=1)

## 5 · Upload a test MP4

Short clip (30s–2min) with clear English speech for the first run.

In [22]:
from google.colab import files
import shutil, os
from pathlib import Path

# ── Optional: set MEDIA_ID for production-style runs ────────────────────────
# In Colab testing, leave empty. When this ports to Modal, the playout PC's
# web-server.js will pass mediaId from media_files DB through to identify
# which row in captions_files / captions_content gets updated.
MEDIA_ID = ''   # e.g., 'm_01KFSCBH7NC7QEF7S2Y4E59NNS'

uploaded = files.upload()
SRC_NAME = list(uploaded.keys())[0]    # original filename, e.g. "commencement.mp4"
SRC_STEM = Path(SRC_NAME).stem          # base, e.g. "commencement"
shutil.move(SRC_NAME, '/content/input.mp4')
print(f'input.mp4 ready - {os.path.getsize("/content/input.mp4"):,} bytes')
print(f'source filename: {SRC_NAME}  (stem: {SRC_STEM})')
if MEDIA_ID:
    print(f'media ID: {MEDIA_ID}')
!ffprobe -v error -show_entries stream=codec_name,codec_type -of default=nw=1 /content/input.mp4

Saving greenscreenOpusClip.mp4 to greenscreenOpusClip.mp4
input.mp4 ready - 2,367,038 bytes
source filename: greenscreenOpusClip.mp4  (stem: greenscreenOpusClip)
codec_name=h264
codec_type=video
codec_name=aac
codec_type=audio


## 6 · Whisper transcribe → VTT (stable-ts, broadcast pacing)

Uses **stable-ts** wrapping faster-whisper. stable-ts auto-regroups long Whisper segments into broadcast-friendly cues using word-level timestamps + punctuation/length splits. We also strip stable-ts's inline word-timestamp tags so the VTT is clean text per cue (otherwise the SEI injector would burn `<00:00:01.420>` into the on-screen captions).

In [23]:
!pip -q install -U "stable-ts[fw]"

In [24]:
import time, re, os
from pathlib import Path

# ── HuggingFace auth (optional, silences rate-limit warning) ────────────────
# To set: Colab left sidebar → 🔑 Secrets → Add HF_TOKEN with your token from
# https://huggingface.co/settings/tokens (read access is sufficient).
# Toggle "Notebook access" on. If unset, downloads still work — just rate-limited.
try:
    from google.colab import userdata
    _hf_tok = userdata.get('HF_TOKEN')
    if _hf_tok:
        os.environ['HF_TOKEN'] = _hf_tok
        os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_tok   # legacy env var
        print('[hf] HF_TOKEN loaded from Colab secrets')
    else:
        print('[hf] no HF_TOKEN — using anonymous (rate-limited) HF downloads')
except Exception:
    pass   # not on Colab or userdata API unavailable — silently skip

import stable_whisper

MODEL = 'large-v3'
DEVICE = 'cuda'
COMPUTE = 'float16'

# ── Known Whisper hallucination phrases (case-insensitive) ──────────────────
HALLUCINATION_PATTERNS = [
    r'^thank(s| you)( so much)?( for watching)?\.?$',
    r'^thanks for watching\.?$',
    r'^thank you\.?$',
    r'^(don\'?t forget to )?(please )?(like and )?subscribe.*$',
    r'^see you (next time|in the next (one|video))\.?$',
    r'^(good)?bye[ !.]*$',
    r'^\[?(music|applause|laughter|sounds?|silence)\]?\.?$',
    r'^[\(\[]?music[\)\]]?$',
    r'^[♪♩♪♫♬\s.]+$',
    r'^you\.?$',
    r'^\.{1,3}$',
    r'^[\s.]*$',
    r'^translated by .*$',
    r'^transcribed by .*$',
    r'^subtitles by .*$',
    r'^captions? by .*$',
]
_HALL_RE = [re.compile(p, re.IGNORECASE) for p in HALLUCINATION_PATTERNS]

def is_hallucination(text):
    t = text.strip()
    if not t: return True
    return any(r.match(t) for r in _HALL_RE)

# ── Load + transcribe ───────────────────────────────────────────────────────
t0 = time.time()
model = stable_whisper.load_faster_whisper(MODEL, device=DEVICE, compute_type=COMPUTE)
print(f'model loaded in {time.time()-t0:.1f}s')

t0 = time.time()
result = model.transcribe(
    '/content/input.mp4',
    language='en',
    regroup=True,
    word_timestamps=True,
    vad_filter=True,
    vad_parameters={'min_silence_duration_ms': 250},
    condition_on_previous_text=False,
    no_repeat_ngram_size=3,
    suppress_silence=True,
    hallucination_silence_threshold=2.0,
)
print(f'transcribed in {time.time()-t0:.1f}s')

# Drop hallucination segments before split
before = len(result.segments)
result.segments = [s for s in result.segments if not is_hallucination(s.text or '')]
if before - len(result.segments):
    print(f'[hallucination] dropped {before-len(result.segments)} segment(s)')

# Broadcast pacing
result.split_by_length(max_chars=42)
result.split_by_duration(max_dur=3.5)
result.split_by_gap(max_gap=0.4)

result.to_srt_vtt('/content/input.vtt', segment_level=True, word_level=False)

# ── VTT post-process: clean inline tags, dedup, sweep, REASSEMBLE WITH BLANK LINES ──
vtt_path = Path('/content/input.vtt')
vtt = vtt_path.read_text()
vtt = re.sub(r'<\d{1,2}:\d{2}:\d{2}\.\d{3}>', '', vtt)
vtt = re.sub(r' {2,}', ' ', vtt)

blocks, cur = [], []
for line in vtt.splitlines():
    if line.strip() == '':
        if cur: blocks.append(cur); cur = []
    else:
        cur.append(line)
if cur: blocks.append(cur)

if not blocks or not blocks[0] or not blocks[0][0].lstrip().upper().startswith('WEBVTT'):
    blocks.insert(0, ['WEBVTT'])

header, *cue_blocks = blocks

def _norm(t): return re.sub(r'[^a-z ]', '', t.lower()).strip()

kept, prev_norm = [], ''
hall_dropped = dup_dropped = 0
for b in cue_blocks:
    if len(b) < 2: continue
    text = ' '.join(b[1:]).strip()
    if is_hallucination(text):
        print(f'[hallucination] dropping post-VTT cue: "{text[:60]}"')
        hall_dropped += 1
        continue
    norm = _norm(text)
    if norm and prev_norm and (norm in prev_norm or prev_norm in norm):
        print(f'[dedup] dropping repeat cue: "{text[:60]}"')
        dup_dropped += 1
        continue
    kept.append(b)
    prev_norm = norm

# CRITICAL: rejoin with BLANK LINES between blocks (\n\n), not single \n.
out = '\n\n'.join('\n'.join(b) for b in [header] + kept) + '\n'
vtt_path.write_text(out)

print(f'\nfinal: {len(kept)} cues  (hallucinations: {hall_dropped}, repeats: {dup_dropped})')
print('─' * 60)
!cat /content/input.vtt
print('─' * 60)
print('Sanity check: blank line between WEBVTT and first cue?',
      '\n\n' in vtt_path.read_text()[:30])

model loaded in 13.1s
Detected Language: english


Transcribe: 100%|██████████| 16.02/16.02 [00:01<00:00, 10.54sec/s]
Adjustment: 100%|██████████| 15.99/15.99 [00:00<00:00, 14242.29sec/s]

transcribed in 2.2s
[hallucination] dropped 1 segment(s)
Saved: /content/input.vtt

final: 5 cues  (hallucinations: 0, repeats: 0)
────────────────────────────────────────────────────────────
WEBVTT

00:00:00.960 --> 00:00:04.170
even now i miss that stubborn goat now

00:00:04.260 --> 00:00:07.290
go before i start crying into my mead the

00:00:08.800 --> 00:00:11.260
tale is done but her

00:00:11.280 --> 00:00:12.890
spirit lives on every

00:00:13.950 --> 00:00:15.390
time the wind howls from the north
────────────────────────────────────────────────────────────
Sanity check: blank line between WEBVTT and first cue? True


## 6.5 · QC + multi-format export (VTT / SRT / SCC)

Runs the broadcast-grade QC pipeline (line wrap to 32 chars, CPS check, duration/overlap/gap fixes, Unicode normalization) on the Whisper VTT. Produces:
- `input.cleaned.vtt` — post-QC VTT (this is what hybridCC-vod injects, so on-screen captions get the QC benefits too)
- `input.srt` — broadcast SRT sidecar
- `input.scc` — Scenarist Closed Caption format for legacy broadcast equipment
- `input.qc.json` — QC report (errors, warnings, fixes applied)

In [25]:
import subprocess
from pathlib import Path

# Install ttconv (CEA-608 / SCC encoder used by vtt_to_scc.py)
print('[install] pip install ttconv...')
subprocess.run(['pip', '-q', 'install', 'ttconv'], check=True)

# vtt_to_scc.py is embedded inline so the notebook is self-contained.
# When the hybridcc repo is on GitHub, swap this for `git clone` + cp.
VTT_TO_SCC_PY = r'''#!/usr/bin/env python3
"""
VTT to SCC Conversion Pipeline with QC
======================================
Uses ttconv for proper CEA-608/SCC encoding via its IMSC canonical model.
Runs pre-conversion QC + normalization, then post-conversion validation.
"""

import argparse
import io
import json
import os
import re
import sys
import textwrap
import unicodedata
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

from ttconv.vtt import reader as vtt_reader
from ttconv.scc import writer as scc_writer
from ttconv.tt import SccWriterConfiguration
from ttconv import model


CEA608_MAX_CHARS_PER_LINE = 32
CEA608_MAX_LINES = 2
MIN_DURATION_SEC = 1.0
MAX_DURATION_SEC = 7.0
MAX_CPS = 20.0
MIN_CPS = 3.0
MIN_GAP_SEC = 0.267
SCC_MIN_LEAD_SEC = 2.5

CEA608_SAFE_PATTERN = re.compile(
    r'[^\x20-\x7E'
    r'áéíóú'
    r'ÁÉÍÓÚ'
    r'çÇ'
    r'ñÑ'
    r'¿¡'
    r'®©°'
    r'½¼¾'
    r'£¢¥'
    r'♪█'
    r'àèìòù'
    r'ÀÈÌÒÙ'
    r'âêîôû'
    r'ÂÊÎÔÛ'
    r'äëïöü'
    r'ÄËÏÖÜ'
    r'åÅ'
    r'æÆ'
    r'øØ'
    r'ß'
    r'\n]'
)

UNICODE_REPLACEMENTS = {
    "‘": "'",
    "’": "'",
    "“": '"',
    "”": '"',
    "–": "-",
    "—": "--",
    "…": "...",
    "​": "",
    " ": " ",
    "﻿": "",
    "‎": "",
    "‏": "",
}


@dataclass
class QCIssue:
    caption_index: int
    timestamp: str
    severity: str
    category: str
    message: str
    auto_fixed: bool = False

@dataclass
class CaptionData:
    index: int
    begin: float
    end: float
    text: str
    lines: list

@dataclass
class QCReport:
    input_file: str
    output_file: str
    total_captions: int
    issues: list = field(default_factory=list)
    pre_qc_issues: list = field(default_factory=list)
    post_qc_issues: list = field(default_factory=list)
    auto_fixes_applied: int = 0

    @property
    def error_count(self):
        return sum(1 for i in self.issues if i.severity == "error")

    @property
    def unfixed_error_count(self):
        return sum(1 for i in self.issues if i.severity == "error" and not i.auto_fixed)

    @property
    def warning_count(self):
        return sum(1 for i in self.issues if i.severity == "warning")

    def to_dict(self):
        d = asdict(self)
        d["error_count"] = self.error_count
        d["remaining_errors"] = len(getattr(self, "remaining_errors", []))
        d["warning_count"] = self.warning_count
        d["pass"] = len(getattr(self, "remaining_errors", [])) == 0 and len(self.post_qc_issues) == 0
        return d


def parse_vtt_to_captions(vtt_path):
    with open(vtt_path, "r", encoding="utf-8") as f:
        doc = vtt_reader.to_model(f)

    captions = []
    body = doc.get_body()
    if body is None:
        return doc, captions

    idx = 0
    for div in body:
        for p in div:
            begin = p.get_begin()
            end = p.get_end()
            if begin is None or end is None:
                continue

            text_parts = []
            for node in p.dfs_iterator():
                if isinstance(node, model.Text):
                    text_parts.append(node.get_text())

            text = "".join(text_parts)
            lines = text.split("\n")

            captions.append(CaptionData(
                index=idx,
                begin=float(begin),
                end=float(end),
                text=text,
                lines=lines
            ))
            idx += 1

    return doc, captions


def format_tc(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def run_pre_qc(captions, report):
    prev = None
    for cap in captions:
        tc = format_tc(cap.begin)
        duration = cap.end - cap.begin

        for i, line in enumerate(cap.lines):
            if len(line) > CEA608_MAX_CHARS_PER_LINE:
                report.pre_qc_issues.append(QCIssue(
                    caption_index=cap.index, timestamp=tc, severity="error",
                    category="line_length",
                    message=f"Line {i+1} is {len(line)} chars (max {CEA608_MAX_CHARS_PER_LINE})"
                ))

        if len(cap.lines) > CEA608_MAX_LINES:
            report.pre_qc_issues.append(QCIssue(
                caption_index=cap.index, timestamp=tc, severity="error",
                category="line_count",
                message=f"Caption has {len(cap.lines)} lines (max {CEA608_MAX_LINES})"
            ))

        if duration < MIN_DURATION_SEC:
            report.pre_qc_issues.append(QCIssue(
                caption_index=cap.index, timestamp=tc, severity="error",
                category="duration_short",
                message=f"Duration {duration:.2f}s below {MIN_DURATION_SEC}s"
            ))
        elif duration > MAX_DURATION_SEC:
            report.pre_qc_issues.append(QCIssue(
                caption_index=cap.index, timestamp=tc, severity="warning",
                category="duration_long",
                message=f"Duration {duration:.2f}s exceeds {MAX_DURATION_SEC}s"
            ))

        clean_text = cap.text.replace("\n", " ")
        char_count = len(clean_text.strip())
        if duration > 0 and char_count > 0:
            cps = char_count / duration
            if cps > MAX_CPS:
                report.pre_qc_issues.append(QCIssue(
                    caption_index=cap.index, timestamp=tc, severity="error",
                    category="reading_speed",
                    message=f"CPS={cps:.1f} exceeds {MAX_CPS}"
                ))
            elif cps < MIN_CPS and char_count > 3:
                report.pre_qc_issues.append(QCIssue(
                    caption_index=cap.index, timestamp=tc, severity="info",
                    category="reading_speed_slow",
                    message=f"CPS={cps:.1f} is very slow"
                ))

        if prev is not None and cap.begin < prev.end:
            overlap = prev.end - cap.begin
            report.pre_qc_issues.append(QCIssue(
                caption_index=cap.index, timestamp=tc, severity="error",
                category="overlap",
                message=f"Overlaps previous by {overlap:.3f}s"
            ))

        if prev is not None and cap.begin >= prev.end:
            gap = cap.begin - prev.end
            if 0 < gap < MIN_GAP_SEC:
                report.pre_qc_issues.append(QCIssue(
                    caption_index=cap.index, timestamp=tc, severity="warning",
                    category="gap_short",
                    message=f"Gap is only {gap:.3f}s (min {MIN_GAP_SEC:.3f}s)"
                ))

        bad_chars = CEA608_SAFE_PATTERN.findall(cap.text)
        if bad_chars:
            unique = set(bad_chars)
            report.pre_qc_issues.append(QCIssue(
                caption_index=cap.index, timestamp=tc, severity="warning",
                category="illegal_chars",
                message=f"Unsupported characters will be stripped: {unique}"
            ))

        if not cap.text.strip():
            report.pre_qc_issues.append(QCIssue(
                caption_index=cap.index, timestamp=tc, severity="warning",
                category="empty",
                message="Empty caption"
            ))

        prev = cap

    report.issues.extend(report.pre_qc_issues)


def normalize_text(text):
    for old, new in UNICODE_REPLACEMENTS.items():
        text = text.replace(old, new)
    text = unicodedata.normalize("NFC", text)
    text = CEA608_SAFE_PATTERN.sub("", text)
    text = re.sub(r" {2,}", " ", text)
    lines = [line.strip() for line in text.split("\n")]
    text = "\n".join(lines)
    return text.strip()


def smart_line_wrap(text, max_width=CEA608_MAX_CHARS_PER_LINE):
    lines = text.split("\n")
    result_lines = []

    for line in lines:
        if len(line) <= max_width:
            result_lines.append(line)
            continue
        wrapped = textwrap.wrap(line, width=max_width, break_long_words=False, break_on_hyphens=True)
        result_lines.extend(wrapped)

    if len(result_lines) > CEA608_MAX_LINES:
        full_text = " ".join(result_lines)
        if len(full_text) <= max_width * CEA608_MAX_LINES:
            mid = len(full_text) // 2
            best_break = mid
            for offset in range(min(15, mid)):
                if mid + offset < len(full_text) and full_text[mid + offset] == " ":
                    best_break = mid + offset
                    break
                if mid - offset >= 0 and full_text[mid - offset] == " ":
                    best_break = mid - offset
                    break
            line1 = full_text[:best_break].strip()
            line2 = full_text[best_break:].strip()
            if len(line1) <= max_width and len(line2) <= max_width:
                result_lines = [line1, line2]
            else:
                result_lines = result_lines[:CEA608_MAX_LINES]
        else:
            result_lines = result_lines[:CEA608_MAX_LINES]

    return "\n".join(result_lines)


def auto_fix_captions(captions, report, apply_fixes=True):
    if not apply_fixes:
        return captions

    fixed = []
    fix_count = 0

    for cap in captions:
        text = cap.text
        begin = cap.begin
        end = cap.end

        if len(fixed) == 0 and begin < SCC_MIN_LEAD_SEC:
            shift = SCC_MIN_LEAD_SEC - begin
            fix_count += 1
            report.issues.append(QCIssue(
                caption_index=cap.index, timestamp=format_tc(begin),
                severity="info", category="scc_lead_in",
                message=f"Shifted first caption from {format_tc(begin)} to {format_tc(SCC_MIN_LEAD_SEC)}",
                auto_fixed=True
            ))
            begin = SCC_MIN_LEAD_SEC
            end = end + shift

        normalized = normalize_text(text)
        if not normalized.strip():
            fix_count += 1
            report.issues.append(QCIssue(
                caption_index=cap.index, timestamp=format_tc(begin),
                severity="info", category="removed_empty",
                message="Removed empty caption", auto_fixed=True
            ))
            continue

        if normalized != text:
            fix_count += 1
            report.issues.append(QCIssue(
                caption_index=cap.index, timestamp=format_tc(begin),
                severity="info", category="normalize",
                message="Unicode normalized", auto_fixed=True
            ))
        text = normalized

        wrapped = smart_line_wrap(text)
        if wrapped != text:
            fix_count += 1
            report.issues.append(QCIssue(
                caption_index=cap.index, timestamp=format_tc(begin),
                severity="info", category="line_wrap",
                message=f"Re-wrapped to fit {CEA608_MAX_CHARS_PER_LINE} chars/line",
                auto_fixed=True
            ))
        text = wrapped

        duration = end - begin
        if duration < MIN_DURATION_SEC and duration > 0:
            end = begin + MIN_DURATION_SEC
            fix_count += 1
            report.issues.append(QCIssue(
                caption_index=cap.index, timestamp=format_tc(begin),
                severity="info", category="duration_extend",
                message=f"Extended duration from {duration:.2f}s to {MIN_DURATION_SEC}s",
                auto_fixed=True
            ))

        new_cap = CaptionData(
            index=cap.index,
            begin=begin,
            end=end,
            text=text,
            lines=text.split("\n")
        )
        fixed.append(new_cap)

    for i in range(1, len(fixed)):
        prev = fixed[i - 1]
        curr = fixed[i]

        if curr.begin < prev.end:
            new_prev_end = curr.begin - MIN_GAP_SEC
            if new_prev_end > prev.begin + 0.5:
                fix_count += 1
                report.issues.append(QCIssue(
                    caption_index=curr.index, timestamp=format_tc(curr.begin),
                    severity="info", category="overlap_fix",
                    message=f"Trimmed previous end to {format_tc(new_prev_end)}",
                    auto_fixed=True
                ))
                prev.end = new_prev_end
            else:
                new_begin = prev.end + MIN_GAP_SEC
                shift = new_begin - curr.begin
                fix_count += 1
                report.issues.append(QCIssue(
                    caption_index=curr.index, timestamp=format_tc(curr.begin),
                    severity="info", category="overlap_fix",
                    message=f"Shifted forward by {shift:.3f}s",
                    auto_fixed=True
                ))
                curr.end += shift
                curr.begin = new_begin
        elif curr.begin >= prev.end:
            gap = curr.begin - prev.end
            if 0 < gap < MIN_GAP_SEC:
                prev.end = curr.begin - MIN_GAP_SEC
                if prev.end < prev.begin + 0.5:
                    prev.end = prev.begin + 0.5
                fix_count += 1
                report.issues.append(QCIssue(
                    caption_index=curr.index, timestamp=format_tc(curr.begin),
                    severity="info", category="gap_fix",
                    message=f"Adjusted gap from {gap:.3f}s to {MIN_GAP_SEC:.3f}s",
                    auto_fixed=True
                ))

    report.auto_fixes_applied = fix_count
    return fixed


def build_model_from_captions(captions):
    doc = model.ContentDocument()
    region = model.Region("r1", doc)
    doc.put_region(region)
    body = model.Body(doc)
    doc.set_body(body)
    div = model.Div(doc)
    body.push_child(div)

    for cap in captions:
        p = model.P(doc)
        p.set_begin(cap.begin)
        p.set_end(cap.end)
        p.set_region(region)
        span = model.Span(doc)
        p.push_child(span)
        text_node = model.Text(doc)
        text_node.set_text(cap.text)
        span.push_child(text_node)
        div.push_child(p)

    return doc


def convert_to_scc(doc, config=None):
    if config is None:
        config = SccWriterConfiguration()
    config.start_tc = "00:00:00;00"
    config.force_popon = True
    config.allow_reflow = True
    return scc_writer.from_model(doc, config)


def validate_scc(scc_content, report):
    lines = scc_content.strip().split("\n")
    if not lines or "Scenarist_SCC" not in lines[0]:
        report.post_qc_issues.append(QCIssue(
            caption_index=-1, timestamp="00:00:00.000",
            severity="error", category="scc_header",
            message="Missing Scenarist_SCC V1.0 header"
        ))

    data_lines = [l for l in lines if l.strip() and "Scenarist_SCC" not in l]
    tc_pattern = re.compile(r"^(\d{2}:\d{2}:\d{2};\d{2})\t(.+)$")

    prev_tc = None
    for line in data_lines:
        match = tc_pattern.match(line)
        if not match:
            continue
        tc_str = match.group(1)
        hex_data = match.group(2)
        hex_words = hex_data.strip().split()
        for word in hex_words:
            if not re.match(r"^[0-9a-fA-F]{4}$", word):
                report.post_qc_issues.append(QCIssue(
                    caption_index=-1, timestamp=tc_str,
                    severity="error", category="scc_hex",
                    message=f"Invalid hex word: {word}"
                ))
        if prev_tc is not None and tc_str <= prev_tc:
            report.post_qc_issues.append(QCIssue(
                caption_index=-1, timestamp=tc_str,
                severity="error", category="scc_tc_order",
                message=f"Timecode not ascending: {tc_str} <= {prev_tc}"
            ))
        prev_tc = tc_str

    if not data_lines:
        report.post_qc_issues.append(QCIssue(
            caption_index=-1, timestamp="00:00:00.000",
            severity="error", category="scc_empty",
            message="SCC file contains no caption data"
        ))

    report.issues.extend(report.post_qc_issues)


def print_report(report):
    print("\n" + "=" * 60)
    print("  VTT to SCC  QC REPORT")
    print("=" * 60)
    print(f"  Input:    {report.input_file}")
    print(f"  Output:   {report.output_file}")
    print(f"  Captions: {report.total_captions}")
    print(f"  Fixes:    {report.auto_fixes_applied}")
    print("-" * 60)

    if report.pre_qc_issues:
        print("\n  PRE-CONVERSION ISSUES:")
        for issue in report.pre_qc_issues:
            icon = {"error": "X", "warning": "!", "info": "i"}.get(issue.severity, "?")
            fixed = " [FIXED]" if issue.auto_fixed else ""
            print(f"    {icon} [{issue.timestamp}] {issue.category}: {issue.message}{fixed}")

    if report.post_qc_issues:
        print("\n  POST-CONVERSION ISSUES:")
        for issue in report.post_qc_issues:
            icon = {"error": "X", "warning": "!", "info": "i"}.get(issue.severity, "?")
            print(f"    {icon} [{issue.timestamp}] {issue.category}: {issue.message}")

    auto_fixed = [i for i in report.issues if i.auto_fixed]
    if auto_fixed:
        print(f"\n  AUTO-FIXES APPLIED: {len(auto_fixed)}")
        for issue in auto_fixed:
            print(f"    + [{issue.timestamp}] {issue.category}: {issue.message}")

    print("\n" + "-" * 60)
    remaining_errs = getattr(report, "remaining_errors", [])
    has_post_errors = any(i.severity == "error" for i in report.post_qc_issues)
    is_pass = len(remaining_errs) == 0 and not has_post_errors
    result = "PASS" if is_pass else "FAIL"

    if remaining_errs:
        print("\n  REMAINING ERRORS (not auto-fixable):")
        for issue in remaining_errs:
            print(f"    X [{issue.timestamp}] {issue.category}: {issue.message}")

    print(f"\n  RESULT: {result}")
    print(f"    {report.error_count} errors found in original")
    print(f"    {report.auto_fixes_applied} auto-fixes applied")
    print(f"    {len(remaining_errs)} errors remaining after fixes")
    print(f"    {report.warning_count} warnings")
    print("=" * 60 + "\n")


def write_cleaned_vtt(captions, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")
        for cap in captions:
            f.write(f"{format_tc(cap.begin)} --> {format_tc(cap.end)}\n")
            f.write(cap.text + "\n\n")


def write_cleaned_srt(captions, path):
    def srt_tc(seconds):
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        ms = int((seconds % 1) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    with open(path, "w", encoding="utf-8") as f:
        for i, cap in enumerate(captions, start=1):
            f.write(f"{i}\n")
            f.write(f"{srt_tc(cap.begin)} --> {srt_tc(cap.end)}\n")
            f.write(cap.text + "\n\n")


def run_pipeline(vtt_path, scc_path, report_path=None,
                 apply_fixes=True, frame_rate="29.97df",
                 cleaned_vtt_path=None, cleaned_srt_path=None):
    report = QCReport(input_file=vtt_path, output_file=scc_path, total_captions=0)

    print(f"[1/6] Parsing VTT: {vtt_path}")
    doc, captions = parse_vtt_to_captions(vtt_path)
    report.total_captions = len(captions)
    print(f"      Found {len(captions)} captions")

    if not captions:
        print("      ERROR: No captions found in VTT file")
        report.issues.append(QCIssue(
            caption_index=-1, timestamp="00:00:00.000",
            severity="error", category="no_captions",
            message="No captions found in input file"
        ))
        print_report(report)
        return report

    print(f"[2/6] Running pre-conversion QC...")
    run_pre_qc(captions, report)
    print(f"      {len(report.pre_qc_issues)} issues found")

    print(f"[3/6] Normalizing captions (auto-fix={'ON' if apply_fixes else 'OFF'})...")
    cleaned = auto_fix_captions(captions, report, apply_fixes)
    print(f"      {report.auto_fixes_applied} fixes applied, {len(cleaned)} captions remaining")

    if apply_fixes:
        post_fix_report = QCReport(input_file=vtt_path, output_file=scc_path, total_captions=len(cleaned))
        run_pre_qc(cleaned, post_fix_report)
        remaining_errors = [i for i in post_fix_report.pre_qc_issues if i.severity == "error"]
        remaining_warnings = [i for i in post_fix_report.pre_qc_issues if i.severity == "warning"]
        report.remaining_errors = remaining_errors
        report.remaining_warnings = remaining_warnings
        if remaining_errors:
            print(f"      {len(remaining_errors)} errors remain after auto-fix")
            report.issues.extend(remaining_errors)
    else:
        report.remaining_errors = [i for i in report.pre_qc_issues if i.severity == "error"]
        report.remaining_warnings = [i for i in report.pre_qc_issues if i.severity == "warning"]

    if cleaned_vtt_path:
        write_cleaned_vtt(cleaned, cleaned_vtt_path)
        print(f"      Cleaned VTT written to {cleaned_vtt_path}")
    if cleaned_srt_path:
        write_cleaned_srt(cleaned, cleaned_srt_path)
        print(f"      Cleaned SRT written to {cleaned_srt_path}")

    print(f"[4/6] Building canonical model...")
    clean_doc = build_model_from_captions(cleaned)

    print(f"[5/6] Converting to SCC (frame rate: {frame_rate})...")
    scc_config = SccWriterConfiguration()
    try:
        scc_output = convert_to_scc(clean_doc, scc_config)
        with open(scc_path, "w", encoding="utf-8") as f:
            f.write(scc_output)
        print(f"      Written to {scc_path}")

        print(f"[6/6] Validating SCC output...")
        validate_scc(scc_output, report)
        print(f"      {len(report.post_qc_issues)} post-conversion issues")

    except RuntimeError as e:
        print(f"      SCC conversion FAILED: {e}")
        report.post_qc_issues.append(QCIssue(
            caption_index=-1, timestamp="00:00:00.000",
            severity="error", category="scc_conversion_failed",
            message=f"ttconv SCC writer error: {e}"
        ))
        report.issues.append(report.post_qc_issues[-1])

    print_report(report)

    if report_path:
        with open(report_path, "w") as f:
            json.dump(report.to_dict(), f, indent=2, default=str)
        print(f"Report saved to {report_path}")

    return report


def main():
    parser = argparse.ArgumentParser(description="VTT to SCC with QC pipeline (ttconv)")
    parser.add_argument("input", help="Input .vtt file")
    parser.add_argument("output", help="Output .scc file")
    parser.add_argument("--report", help="Save QC report as JSON", default=None)
    parser.add_argument("--no-fix", action="store_true", help="Disable auto-fixes (QC report only)")
    parser.add_argument("--frame-rate", default="29.97df", help="SCC frame rate (default: 29.97df)")
    parser.add_argument("--cleaned-vtt", help="Write cleaned (post-QC) VTT to this path", default=None)
    parser.add_argument("--cleaned-srt", help="Write cleaned (post-QC) SRT to this path", default=None)
    args = parser.parse_args()

    if not os.path.exists(args.input):
        print(f"Error: Input file not found: {args.input}")
        sys.exit(1)

    report = run_pipeline(
        vtt_path=args.input,
        scc_path=args.output,
        report_path=args.report,
        apply_fixes=not args.no_fix,
        frame_rate=args.frame_rate,
        cleaned_vtt_path=args.cleaned_vtt,
        cleaned_srt_path=args.cleaned_srt,
    )

    remaining = len(getattr(report, "remaining_errors", []))
    sys.exit(0 if remaining == 0 else 1)


if __name__ == "__main__":
    main()
'''

target = Path('/content/vtt_to_scc.py')
target.write_text(VTT_TO_SCC_PY, encoding='utf-8')
print(f'staged {target} ({target.stat().st_size:,} bytes)')

[install] pip install ttconv...
staged /content/vtt_to_scc.py (23,531 bytes)


In [26]:
import subprocess, json
from pathlib import Path

# Run QC pipeline:
#   - validates the Whisper VTT against CEA-608 broadcast rules
#   - normalizes Unicode, wraps to 32 chars/line, fixes overlap/duration/gap
#   - writes cleaned VTT (this becomes the input to hybridCC-vod)
#   - writes SRT sidecar
#   - writes SCC for legacy broadcast equipment
#   - writes JSON QC report
result = subprocess.run([
    'python', '/content/vtt_to_scc.py',
    '/content/input.vtt',
    '/content/input.scc',
    '--report', '/content/input.qc.json',
    '--cleaned-vtt', '/content/input.cleaned.vtt',
    '--cleaned-srt', '/content/input.srt',
    '--frame-rate', '29.97df',
], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

# Replace input.vtt with the cleaned version so hybridCC-vod (Section 7) uses
# the QC'd captions — better wrap, no overlaps, normalized text.
cleaned = Path('/content/input.cleaned.vtt')
if cleaned.exists():
    Path('/content/input.vtt').write_text(cleaned.read_text())
    print('\n[+] swapped /content/input.vtt → cleaned version (used by hybridCC-vod)')
else:
    print('\n[!] cleaned VTT missing — Section 7 will use raw Whisper VTT')

# Quick QC summary
qc = Path('/content/input.qc.json')
if qc.exists():
    data = json.loads(qc.read_text())
    print(f"\nQC: {data.get('total_captions', 0)} cues, "
          f"{data.get('error_count', 0)} errors original, "
          f"{data.get('auto_fixes_applied', 0)} fixes applied, "
          f"{data.get('remaining_errors', 0)} remaining, "
          f"PASS={data.get('pass', False)}")

# Show all four artifacts
!ls -la /content/input.vtt /content/input.cleaned.vtt /content/input.srt /content/input.scc /content/input.qc.json 2>&1

[1/6] Parsing VTT: /content/input.vtt
      Found 5 captions
[2/6] Running pre-conversion QC...
      6 issues found
[3/6] Normalizing captions (auto-fix=ON)...
      6 fixes applied, 5 captions remaining
      2 errors remain after auto-fix
      Cleaned VTT written to /content/input.cleaned.vtt
      Cleaned SRT written to /content/input.srt
[4/6] Building canonical model...
[5/6] Converting to SCC (frame rate: 29.97df)...
      Written to /content/input.scc
[6/6] Validating SCC output...
      0 post-conversion issues

  VTT to SCC  QC REPORT
  Input:    /content/input.vtt
  Output:   /content/input.scc
  Captions: 5
  Fixes:    6
------------------------------------------------------------

  PRE-CONVERSION ISSUES:
    X [00:00:00.960] line_length: Line 1 is 38 chars (max 32)
    X [00:00:04.259] line_length: Line 1 is 41 chars (max 32)
    ! [00:00:04.259] gap_short: Gap is only 0.090s (min 0.267s)
    ! [00:00:11.279] gap_short: Gap is only 0.020s (min 0.267s)
    X [00:00:13.949

## 7 · Inject CEA-608 via the FLV pipe

In [ ]:
%cd /content
# Pipeline:
#   1st ffmpeg: mp4 -> flv pipe. -bsf strips SEI NALs from input so any
#               pre-existing CEA-608/708 can't double up with ours.
#               (filter_units=remove_types=6 is the ffmpeg 4.4-compatible
#               way; ffmpeg 5.0+ has finer-grained h264_metadata options.)
#   hybridCC-vod: injects fresh CEA-608 SEI by PTS
#   2nd ffmpeg: flv -> mp4 with -c:v copy (the new SEI rides through)
!set -o pipefail; \
  ffmpeg -y -hide_banner -loglevel warning \
    -i input.mp4 \
    -c:v copy \
    -bsf:v "filter_units=remove_types=6" \
    -c:a aac -ac 2 -ar 44100 \
    -f flv pipe:1 \
  | ./hybridCC-vod input.vtt \
  | ffmpeg -y -hide_banner -loglevel warning \
    -f flv -i pipe:0 -c:v copy -c:a copy -movflags +faststart output.mp4 \
  && ls -la output.mp4 || echo 'pipeline error'

## 8 · Verify CEA-608 landed

ffprobe's `closed_captions=` flag is sometimes a false-negative when the SEI is present but not in the analyzed segment. The real proof is the round-trip extract via `[out0+subcc]` — if it produces text, the SEI is there.

In [28]:
print('=== ffprobe stream summary ===')
!ffprobe -v error -show_streams -select_streams v:0 /content/output.mp4 | grep -E 'codec_name|width|height|closed_captions|duration='
print('\n(closed_captions=0 here is often a false-negative — see extraction below)\n')

print('=== Extract CEA-608 from output (first 30s) ===')
!ffmpeg -hide_banner -loglevel error -t 30 -f lavfi -i "movie=/content/output.mp4[out0+subcc]" -map 0:s -c:s webvtt -y /content/extracted.vtt 2>&1 || echo 'extraction failed (try a longer clip)'
!head -30 /content/extracted.vtt 2>/dev/null || echo 'no extracted.vtt produced'

=== ffprobe stream summary ===
/content/output.mp4: No such file or directory

(closed_captions=0 here is often a false-negative — see extraction below)

=== Extract CEA-608 from output (first 30s) ===
[Parsed_movie_0 @ 0x567276c72fc0] Failed to avformat_open_input '/content/output.mp4'
[lavfi @ 0x567276c715c0] Error initializing filter 'movie' with args '/content/output.mp4'
movie=/content/output.mp4[out0+subcc]: No such file or directory
extraction failed (try a longer clip)
no extracted.vtt produced


## 8.5 · Independent proof report (the customer-facing artifact)

Two independent third-party CEA-608 decoders read the captions back from `output.mp4`. If both produce text that matches the source VTT, the SEI bytes are unambiguously valid — same proof a broadcast engineer would produce for FCC §79.1 acceptance.

Outputs:
- `proof-report.html` — branded HTML, side-by-side decoded text, PASS/FAIL verdict
- `verify.txt` — plaintext "how to verify yourself in 60 seconds"

Both ship with every CC delivery so the customer can confirm without trusting our word.

In [29]:
import subprocess, re, json, os, html
from pathlib import Path
from datetime import datetime

# ── Install verification tools ──────────────────────────────────────────────
!apt-get -qq install -y ccextractor libpango-1.0-0 libpangoft2-1.0-0 libcairo2 2>&1 | tail -2
subprocess.run(['pip', '-q', 'install', 'weasyprint'], check=True)
ccextractor_bin = subprocess.run(['which', 'ccextractor'], capture_output=True, text=True).stdout.strip()

OUTPUT_MP4 = '/content/output.mp4'
SOURCE_VTT = '/content/input.vtt'
SCC_PATH   = '/content/input.scc'
QC_PATH    = '/content/input.qc.json'

# Pull source identity from Section 5 globals (filename + optional mediaId)
src_name  = globals().get('SRC_NAME', 'output.mp4')
src_stem  = globals().get('SRC_STEM', 'output')
media_id  = globals().get('MEDIA_ID', '') or ''

# ── Decoder #1: ffmpeg [out0+subcc] ────────────────────────────────────────
print('[1/3] decoder #1 extraction...')
ff_path = '/content/_extracted_ffmpeg.vtt'
subprocess.run(
    ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
     '-f', 'lavfi', '-i', f'movie={OUTPUT_MP4}[out0+subcc]',
     '-map', '0:s', '-c:s', 'webvtt', ff_path],
    capture_output=True, text=True
)
ff_ok = Path(ff_path).exists() and Path(ff_path).stat().st_size > 50

# ── Decoder #2: CCExtractor ────────────────────────────────────────────────
print('[2/3] decoder #2 extraction...')
cc_path = '/content/_extracted_ccextractor.srt'
if ccextractor_bin:
    subprocess.run([ccextractor_bin, OUTPUT_MP4, '-o', cc_path],
                   capture_output=True, text=True)
cc_ok = Path(cc_path).exists() and Path(cc_path).stat().st_size > 50

# ── Decoder #3: SCC format check ──────────────────────────────────────────
print('[3/3] SCC format check...')
scc_ok = False
scc_first = ''
if Path(SCC_PATH).exists() and Path(SCC_PATH).stat().st_size > 0:
    scc_first = Path(SCC_PATH).read_text().splitlines()[0]
    scc_ok = scc_first.startswith('Scenarist_SCC')

# ── Word-overlap scoring ───────────────────────────────────────────────────
def to_word_set(text):
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'\d{1,2}:\d{2}:\d{2}[.,]\d+\s*-->\s*\d{1,2}:\d{2}:\d{2}[.,]\d+', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.M)
    text = re.sub(r'WEBVTT.*', '', text)
    return set(re.findall(r'[a-z]{2,}', text.lower()))

def jaccard(a, b):
    return len(a & b) / max(len(a | b), 1) if (a or b) else 0.0

source_words = to_word_set(Path(SOURCE_VTT).read_text())
ff_words     = to_word_set(Path(ff_path).read_text()) if ff_ok else set()
cc_words     = to_word_set(Path(cc_path).read_text()) if cc_ok else set()

ff_score = jaccard(source_words, ff_words)
cc_score = jaccard(source_words, cc_words)

PASS_THRESHOLD = 0.80
qc = json.loads(Path(QC_PATH).read_text()) if Path(QC_PATH).exists() else {}
qc_pass = bool(qc.get('pass', False))

# ── Pull remaining-error DETAILS from qc.json ─────────────────────────────
def get_remaining_errors(qc_data):
    rem = qc_data.get('remaining_errors')
    if isinstance(rem, list):
        return rem
    rem_count = rem if isinstance(rem, int) else 0
    if rem_count <= 0:
        return []
    issues = qc_data.get('issues', [])
    error_issues = [i for i in issues if i.get('severity') == 'error']
    return error_issues[-rem_count:]

remaining_errors = get_remaining_errors(qc)
auto_fixed = [i for i in qc.get('issues', []) if i.get('auto_fixed')]

encoding_pass = (ff_score >= PASS_THRESHOLD) and (cc_score >= PASS_THRESHOLD) and scc_ok
overall_pass = encoding_pass and qc_pass

# ── HTML proof report ─────────────────────────────────────────────────────
def status_badge(ok):
    return '<span class="pass">&#10003; PASS</span>' if ok else '<span class="fail">&#10007; FAIL</span>'

def safe(s, n=2000):
    return html.escape((s or '')[:n])

src_text = Path(SOURCE_VTT).read_text()
ff_text  = Path(ff_path).read_text() if ff_ok else '(not produced)'
cc_text  = Path(cc_path).read_text() if cc_ok else '(ccextractor unavailable or failed)'

verdict_class = 'pass' if overall_pass else 'fail'
verdict_text  = 'PASS' if overall_pass else 'FAIL'

errors_html = ''
if remaining_errors:
    rows = []
    for e in remaining_errors:
        rows.append(f"""<tr>
<td><code>{html.escape(e.get('timestamp','-'))}</code></td>
<td>cue #{e.get('caption_index','-')}</td>
<td><code>{html.escape(e.get('category','-'))}</code></td>
<td>{html.escape(e.get('message','-'))}</td>
</tr>""")
    errors_html = f"""
<h2>QC errors that need attention ({len(remaining_errors)})</h2>
<p class="muted">These survived the auto-fix pass and may need manual editing for strict broadcast compliance.</p>
<table>
<tr><th>Timestamp</th><th>Cue</th><th>Category</th><th>Message</th></tr>
{''.join(rows)}
</table>
"""

fixes_html = ''
if auto_fixed:
    rows = []
    for f in auto_fixed[:20]:
        rows.append(f"""<tr>
<td><code>{html.escape(f.get('timestamp','-'))}</code></td>
<td><code>{html.escape(f.get('category','-'))}</code></td>
<td>{html.escape(f.get('message','-'))}</td>
</tr>""")
    more = f"<p class='muted'>... and {len(auto_fixed)-20} more</p>" if len(auto_fixed) > 20 else ''
    fixes_html = f"""
<h2>Auto-fixes applied ({len(auto_fixed)})</h2>
<table>
<tr><th>Timestamp</th><th>Category</th><th>What was fixed</th></tr>
{''.join(rows)}
</table>
{more}
"""

# Identity row in the header
identity_lines = [f'Source: <code>{html.escape(src_name)}</code>']
if media_id:
    identity_lines.append(f'Media&nbsp;ID: <code>{html.escape(media_id)}</code>')
identity_html = ' &middot; '.join(identity_lines)

html_doc = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<title>HybridCC Caption Proof Report - {html.escape(src_name)}</title>
<style>
@page {{ size: letter; margin: 0.75in; }}
body {{ font-family: -apple-system, system-ui, "Segoe UI", sans-serif; max-width: 900px; margin: 2em auto; padding: 0 1em; line-height: 1.5; color: #222; }}
h1 {{ border-bottom: 2px solid #222; padding-bottom: .3em; }}
.pass {{ color: #0a7; font-weight: bold; }}
.fail {{ color: #c33; font-weight: bold; }}
.badge {{ display: inline-block; padding: .4em 1em; border-radius: .3em; color: white; font-size: 1.2em; font-weight: bold; }}
.badge.pass {{ background: #0a7; }}
.badge.fail {{ background: #c33; }}
table {{ border-collapse: collapse; width: 100%; margin: 1em 0; }}
th, td {{ padding: .6em; border: 1px solid #ddd; text-align: left; vertical-align: top; }}
th {{ background: #f5f5f5; }}
pre {{ background: #f7f7f7; padding: .8em; overflow-x: auto; font-size: .85em; max-height: 250px; white-space: pre-wrap; word-break: break-word; }}
.muted {{ color: #777; font-size: .9em; }}
.center {{ text-align: center; }}
.cols {{ display: grid; grid-template-columns: 1fr 1fr; gap: 1em; }}
@media (max-width: 700px) {{ .cols {{ grid-template-columns: 1fr; }} }}

@media print {{
  body {{ margin: 0; max-width: none; }}
  pre {{ max-height: none; overflow: visible; font-size: .75em; page-break-inside: auto; }}
  table {{ page-break-inside: avoid; }}
  h2 {{ page-break-after: avoid; }}
  .cols {{ grid-template-columns: 1fr 1fr; }}
  a {{ color: #222; text-decoration: none; }}
}}
</style>
</head><body>

<h1>HybridCC Caption Proof Report</h1>
<p class="muted">Generated {datetime.now().isoformat(timespec='seconds')}<br>{identity_html}</p>

<h2>Verdict: <span class="badge {verdict_class}">{verdict_text}</span></h2>

<table>
<tr><th>Check</th><th>Result</th><th>Score</th></tr>
<tr><td>CEA-608 captions in MP4 &mdash; decoder #1</td><td>{status_badge(ff_score >= PASS_THRESHOLD)}</td><td>{ff_score*100:.1f}% word match vs source</td></tr>
<tr><td>CEA-608 captions in MP4 &mdash; decoder #2</td><td>{status_badge(cc_score >= PASS_THRESHOLD)}</td><td>{cc_score*100:.1f}% word match vs source</td></tr>
<tr><td>SCC sidecar format</td><td>{status_badge(scc_ok)}</td><td>{html.escape(scc_first or 'no header found')}</td></tr>
<tr><td>Pre-encode QC (FCC &sect;79.1 line/CPS/duration rules)</td><td>{status_badge(qc_pass)}</td><td>{qc.get('total_captions', 0)} cues &middot; {len(auto_fixed)} auto-fixes &middot; {len(remaining_errors)} remaining errors</td></tr>
</table>

<h2>What this means</h2>
<p>The <strong>encoding is verified</strong> &mdash; two independent open-source decoders read the captions out of <code>{html.escape(src_name)}</code> with {min(ff_score, cc_score)*100:.0f}%+ word match, and the SCC sidecar carries a valid Scenarist_SCC V1.0 header. This is the same evidence a broadcast engineer would file for FCC &sect;79.1 acceptance.</p>
{'<p><strong>QC flagged ' + str(len(remaining_errors)) + ' issue(s) that the auto-fixer could not resolve.</strong> See "QC errors that need attention" below for details. The captions still aired correctly, but these may need manual editing for strict broadcast compliance.</p>' if remaining_errors else ''}
<p>You can reproduce these results yourself with publicly available tools &mdash; see <code>verify.txt</code>.</p>

{errors_html}

{fixes_html}

<h2>Side-by-side text comparison</h2>
<div class="cols">
<div><h3>Source (post-QC VTT)</h3><pre>{safe(src_text)}</pre></div>
<div><h3>Decoder #1 extraction</h3><pre>{safe(ff_text)}</pre></div>
</div>
<div class="cols">
<div><h3>Decoder #2 extraction</h3><pre>{safe(cc_text)}</pre></div>
<div><h3>SCC header</h3><pre>{html.escape(Path(SCC_PATH).read_text()[:1500] if Path(SCC_PATH).exists() else '(no SCC produced)')}</pre></div>
</div>

<h2>Verify it yourself (60 seconds)</h2>
<ol>
<li><strong>Playback:</strong> Open <code>{html.escape(src_name)}</code> in VLC. Subtitle menu &rarr; Closed Captions &rarr; CC1. Text appears.</li>
<li><strong>ffmpeg:</strong> <code>ffmpeg -i "movie={html.escape(src_name)}[out0+subcc]" -map 0:s -c:s webvtt out.vtt</code></li>
<li><strong>CCExtractor:</strong> <code>ccextractor {html.escape(src_name)} -o out.srt</code></li>
<li><strong>SCC format:</strong> Open <code>{html.escape(src_stem)}.scc</code> in any text editor. First line: <code>Scenarist_SCC V1.0</code>.</li>
</ol>

<p class="muted center">HybridCC &middot; FCC &sect;79.1 / &sect;79.4 compliant</p>

</body></html>"""

Path('/content/proof-report.html').write_text(html_doc, encoding='utf-8')

# ── PDF render via WeasyPrint ─────────────────────────────────────────────
try:
    from weasyprint import HTML as WeasyHTML
    WeasyHTML(filename='/content/proof-report.html').write_pdf('/content/proof-report.pdf')
    pdf_size = Path('/content/proof-report.pdf').stat().st_size
    print(f'proof-report.pdf  {pdf_size:,} bytes')
except Exception as e:
    print(f'[!] PDF render failed: {e}')

# ── Plaintext verify.txt — ASCII-ONLY (Windows-safe) ──────────────────────
err_block = ''
if remaining_errors:
    err_lines = ['', 'QC ERRORS THAT NEED ATTENTION', '-' * 60]
    for e in remaining_errors:
        err_lines.append(f"  [{e.get('timestamp','-')}] cue#{e.get('caption_index','-')} {e.get('category','-')}: {e.get('message','-')}")
    err_block = '\n'.join(err_lines) + '\n'

media_line = f"Media ID: {media_id}\n" if media_id else ""

verify_txt = (
f"""HYBRIDCC CAPTION PROOF -- {src_name}
{media_line}Generated: {datetime.now().isoformat(timespec='seconds')}

VERDICT: {verdict_text}
============================================================

CHECK                           RESULT      SCORE
ffmpeg subcc decoder            {'PASS' if ff_score >= PASS_THRESHOLD else 'FAIL':4}        {ff_score*100:5.1f}% word match
CCExtractor decoder             {'PASS' if cc_score >= PASS_THRESHOLD else 'FAIL':4}        {cc_score*100:5.1f}% word match
SCC format header               {'PASS' if scc_ok else 'FAIL':4}        {scc_first or '(no header)'}
Pre-encode QC                   {'PASS' if qc_pass else 'FAIL':4}        {len(auto_fixed)} fixes, {len(remaining_errors)} remaining errors
{err_block}
VERIFY YOURSELF IN 60 SECONDS -- no special tools needed
============================================================

1. PLAYBACK CHECK (anyone with VLC):
   Open {src_name}. Subtitle -> Closed Captions -> CC1. Text appears.

2. EXTRACTION CHECK (ffmpeg):
   ffmpeg -i "movie={src_name}[out0+subcc]" -map 0:s -c:s webvtt out.vtt
   out.vtt should match {src_stem}.vtt.

3. INDEPENDENT THIRD-PARTY DECODER (CCExtractor -- free, open source):
   https://github.com/CCExtractor/ccextractor
   ccextractor {src_name} -o out.srt
   out.srt should contain the captions text.

4. SCC FORMAT CHECK (legacy broadcast equipment):
   Open {src_stem}.scc in any text editor.
   First line must be: Scenarist_SCC V1.0

5. QC AUDIT TRAIL:
   Open {src_stem}.proof.pdf -- every cue, every check, every fix logged.

Two open-source decoders agreeing on the text = proof the captions are
correctly encoded. This is the same evidence a broadcast engineer files
for FCC Sec.79.1 acceptance.

HybridCC -- FCC Sec.79.1 / Sec.79.4 compliant
"""
)

Path('/content/verify.txt').write_text(verify_txt, encoding='ascii', errors='replace')

print('=' * 60)
print(verify_txt)
print('=' * 60)
print(f'proof-report.html  {Path("/content/proof-report.html").stat().st_size:,} bytes')
print(f'verify.txt         {Path("/content/verify.txt").stat().st_size:,} bytes')

[1/3] decoder #1 extraction...
[2/3] decoder #2 extraction...
[3/3] SCC format check...


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.007s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'glyf' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'glyf' tabl

proof-report.pdf  21,533 bytes
HYBRIDCC CAPTION PROOF -- greenscreenOpusClip.mp4
Generated: 2026-05-07T05:37:20

VERDICT: FAIL

CHECK                           RESULT      SCORE
ffmpeg subcc decoder            FAIL          0.0% word match
CCExtractor decoder             FAIL          0.0% word match
SCC format header               PASS        Scenarist_SCC V1.0
Pre-encode QC                   FAIL        6 fixes, 2 remaining errors

QC ERRORS THAT NEED ATTENTION
------------------------------------------------------------
  [00:00:02.500] cue#0 reading_speed: CPS=25.5 exceeds 20.0
  [00:00:13.949] cue#4 reading_speed: CPS=23.6 exceeds 20.0

VERIFY YOURSELF IN 60 SECONDS -- no special tools needed

1. PLAYBACK CHECK (anyone with VLC):
   Open greenscreenOpusClip.mp4. Subtitle -> Closed Captions -> CC1. Text appears.

2. EXTRACTION CHECK (ffmpeg):
   ffmpeg -i "movie=greenscreenOpusClip.mp4[out0+subcc]" -map 0:s -c:s webvtt out.vtt
   out.vtt should match greenscreenOpusClip.vtt.

3. IN

## 9 · Download captioned MP4 + source VTT

In [ ]:
from google.colab import files
import os, shutil, zipfile
from pathlib import Path

stem = globals().get("SRC_STEM", "output")

# Sanity check - Section 7 must have produced output.mp4
if not os.path.exists("/content/output.mp4"):
    print("[!] /content/output.mp4 not found.")
    print("    Section 7 (Inject CEA-608 via the FLV pipe) did not run successfully.")
    print("    Re-run Section 7 first; check its output for ffmpeg errors.")
    raise SystemExit("Pipeline incomplete - cannot bundle download.")

# 1. Captioned MP4 - same name as source, replaces in media folder
mp4_dst = f"/content/{stem}.mp4"
shutil.copy("/content/output.mp4", mp4_dst)

# 2. VTT - goes into captions_content DB row in production
vtt_dst = f"/content/{stem}.vtt"
shutil.copy("/content/input.vtt", vtt_dst)

# 3. Audit packet - bundle the rest (sidecars + proof artifacts) into one zip
audit_zip = f"/content/{stem}.audit.zip"
audit_files = [
    ("/content/input.srt",          f"{stem}.srt"),
    ("/content/input.scc",          f"{stem}.scc"),
    ("/content/input.qc.json",      f"{stem}.qc.json"),
    ("/content/proof-report.pdf",   f"{stem}.proof.pdf"),
    ("/content/proof-report.html",  f"{stem}.proof.html"),
    ("/content/verify.txt",         f"{stem}.verify.txt"),
]
with zipfile.ZipFile(audit_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for src, name_in_zip in audit_files:
        if os.path.exists(src):
            z.write(src, name_in_zip)
            print(f"  zipped: {name_in_zip}  ({os.path.getsize(src):,} bytes)")
        else:
            print(f"  skipped: {name_in_zip} (source missing)")

# Three downloads: MP4 (media folder), VTT (DB ingest), audit zip (customer reference)
for path in (mp4_dst, vtt_dst, audit_zip):
    print(f"\nDownloading {Path(path).name}  ({os.path.getsize(path):,} bytes)")
    files.download(path)

## Next steps

- **Modal port** — same image add-ons, same compile, drop into `modal-caption-server.py` as an `InjectWorker` class with a new `POST /caption/inject` endpoint.
- **Hetzner live add-on** — `src/legacy/hybridCC-stdin.c` builds the same way; same notebook minus Whisper, plug into MediaMTX.
- **Windows build** — MSVC 2019 + libcaption's CMakeLists. Same source, the `#ifdef _WIN32` blocks reactivate the `_setmode` patch.
- **Format options** — replace `-a53cc 1` with `-c:s mov_text` for a web-friendly VTT track instead of (or alongside) CEA-608 SEI.